# ARA Opening Signature Research — v3 Two-Lane Evaluation

Notebook ini melanjutkan v2.1, tetapi mengubah pendekatan evaluasi dari **linear blended score** menjadi **two-lane / OR strategy**:

1. **Model-score lane**: menangkap event seperti DIVA yang tertangkap oleh `score_ara`.
2. **Behavioral-ignition lane**: menangkap event seperti INTD/TALF/LCKM yang lebih terlihat dari `ara_signature_score_v1`.

Alasan perubahan: hasil v2 menunjukkan single blended score 50/50, 60/40, 70/30 justru bisa menurunkan ranking event behavioral karena `score_ara` rendah.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

OUTPUT_DIR = Path("ara_opening_signature_research_v3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ubah path ini jika output v2 ada di lokasi lain.
CANDIDATE_PATHS = [
    Path("ara_opening_signature_research_v2_outputs/ara_opening_signature_dataset_v2.csv"),
    Path("ara_opening_signature_dataset_v2.csv"),
    Path("/mnt/data/ara_opening_signature_dataset_v2.csv"),
]

DATASET_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError("Tidak menemukan ara_opening_signature_dataset_v2.csv. Letakkan file tersebut di folder notebook atau ara_opening_signature_research_v2_outputs/.")

df = pd.read_csv(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
if "next_date" in df.columns:
    df["next_date"] = pd.to_datetime(df["next_date"])

print("Loaded:", DATASET_PATH)
print(df.shape)
display(df.head())

Loaded: ara_opening_signature_research_v2_outputs/ara_opening_signature_dataset_v2.csv
(1165, 269)


,date,ticker,rank1_buyer,rank1_seller,rank1_buyer_type,rank1_seller_type,market_regime,open,high,low,close,volume,value,frequency,foreign_buy,foreign_sell,ret_1d,ret_5d,ret_10d,ret_20d,ma_5,ma_20,volume_ma20,volatility_20d,close_vs_ma20,volume_ratio_20d,traded_value_proxy,has_broksum,buy_val_total,sell_val_total,net_val_total,buy_lot_total,sell_lot_total,buy_freq_total,sell_freq_total,buy_val_total_sane,sell_val_total_sane,net_val_total_sane,rank1_buy_val_sane,rank1_sell_val_sane,net_flow_ratio,net_buy_flag,buyer_dominance_ratio,seller_dominance_ratio,broker_value_anomaly_flag,rank1_same_buyer_flag,rank1_same_buyer_streak,rank1_buyer_daily_count,rank1_buyer_daily_share,rank1_buyer_overcrowded_flag,insider_event_count,insider_buy_count,insider_sell_count,insider_net_shares,insider_net_pct_sum,insider_foreign_event_count,insider_local_event_count,has_insider_activity,has_corporate_action,ca_event_count,...,score_momentum_10d,score_momentum_20d,score_scalp,score_swing,score_position,next_date,next_open,next_high,next_low,next_close,next_volume,next_value,next_traded_value_proxy,ara_limit_pct,tick_size,raw_ara_price,ara_price_tick_adjusted,tick_distance_to_ara_open,next_open_ret,ara_ratio,valid_opening_label,is_full_ara_tick_adjusted,ara_opening_class,is_strong_gap_or_better,is_near_or_full_ara,is_full_ara_opening,rank_volume_ratio_20d,rank_ret_1d,rank_ret_5d,rank_ret_10d,rank_close_vs_ma20,rank_volatility_20d,rank_buyer_dominance_ratio,rank_net_flow_ratio,rank_rank1_same_buyer_streak,rank_buy_val_total,rank_net_val_total,rank_traded_value_proxy,rank_score_sm,rank_score_ara,rank_score_mm_silent,rank_score_momentum_5d__momentum_ranker__xgb__momentum_5d,rank_score_momentum_10d__momentum_ranker__hgb__momentum_10d,rank_score_scalp__multi_strategy_time__rank_hgb__scalp,rank_score_swing__multi_strategy_time__hgb__swing,rank_score_position__multi_strategy_time__xgb__position,rank_score_momentum_5d,rank_score_momentum_10d,rank_score_momentum_20d,rank_score_scalp,rank_score_swing,rank_score_position,ara_signature_score_v1,rank_ara_signature_score_v1,final_ara_watch_score_50_50,final_ara_watch_score_60_40,final_ara_watch_score_70_30,rank_final_ara_watch_score_50_50,rank_final_ara_watch_score_60_40,rank_final_ara_watch_score_70_30
0,2026-05-13,ACES,YU,YU,ASING,ASING,risk_off,374.0,374.0,368.0,370.0,17708500.0,6.566484e+09,2018.0,877600.0,7634400.0,-0.041451,0.005435,-0.060914,0.033520,374.4,372.5,43161765.0,0.025151,-0.006711,0.410282,6.566484e+09,1,1.019886e+10,1.022573e+10,NaN,274676.00,275401.00,1981.0,2014.0,1.019886e+10,1.022573e+10,-26869000.0,2.788198e+09,4.408606e+09,-0.001316,0,0.273383,0.431129,0,0,1,14,0.016204,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,...,0.185502,0.078549,0.077585,0.111710,0.146365,2026-05-18,370.0,370.0,348.0,354.0,80858000.0,2.878577e+10,2.878577e+10,0.25,2,462.5,462.0,46.0,0.000000,0.000000,True,0,normal_movement,0,0,0,0.336538,0.495192,0.740385,0.581731,0.682692,0.269231,0.432692,0.187500,0.274038,0.711538,NaN,0.629808,NaN,0.264423,NaN,NaN,NaN,NaN,NaN,NaN,0.201923,0.254808,0.235577,0.081731,0.201923,0.245192,0.453173,0.461538,0.362981,0.343269,0.323558,0.144231,0.163462,0.206731
1,2026-05-18,ACES,YP,DR,ASING,ASING,risk_off,370.0,370.0,348.0,354.0,80858000.0,2.878577e+10,5711.0,7028300.0,16358200.0,-0.043243,-0.011173,-0.106061,0.011429,373.6,372.7,44425605.0,0.026605,-0.050174,1.820077,2.878577e+10,1,2.819248e+10,2.873469e+10,NaN,791842.00,807148.00,5517.0,5665.0,2.819248e+10,2.873469e+10,-542215200.0,6.748923e+09,1.651788e+10,-0.009525,0,0.239387,0.574841,0,0,1,44,0.050926,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,...,0.240408,0.095112,0.102821,0.125219,0.168672,2026-05-19,356.0,358.0,344.0,346.0,38818500.0,1.354934e+10,1.354934e+10,0.25,2,442.5,442.0,43.0,0.005650,0.022599,True,0,normal_movement,0,0,0,0.815534,0.281553,0.669903,0.441748,0.582524,0.315534,0.344660,0.053398,0.269417,0.844660,NaN,0.854369,NaN,0.252427,NaN,NaN,NaN,NaN,NaN,NaN,0.291262,0.490291,0.349515,0.184466,0.291262,0.427184,0.483083,0.500000,0.37

## 1. Validate target/event columns

In [3]:
df = df.copy()

event_classes = {"strong_gap_up", "near_ara", "full_ara_opening"}
near_classes = {"near_ara", "full_ara_opening"}

df["is_strong_gap_or_better"] = df["ara_opening_class"].isin(event_classes).astype(int)
df["is_near_or_full_ara"] = df["ara_opening_class"].isin(near_classes).astype(int)

summary = {
    "rows": int(len(df)),
    "unique_dates": int(df["date"].nunique()),
    "unique_tickers": int(df["ticker"].nunique()),
    "class_distribution": df["ara_opening_class"].value_counts(dropna=False).to_dict(),
    "strong_gap_or_better_events": int(df["is_strong_gap_or_better"].sum()),
    "near_or_full_ara_events": int(df["is_near_or_full_ara"].sum()),
    "baseline_strong_gap_rate": float(df["is_strong_gap_or_better"].mean()),
    "baseline_near_or_full_ara_rate": float(df["is_near_or_full_ara"].mean()),
}
print(json.dumps(summary, indent=2, default=str))

{
  "rows": 1165,
  "unique_dates": 6,
  "unique_tickers": 259,
  "class_distribution": {
    "normal_movement": 1160,
    "strong_gap_up": 4,
    "near_ara": 1
  },
  "strong_gap_or_better_events": 5,
  "near_or_full_ara_events": 1,
  "baseline_strong_gap_rate": 0.004291845493562232,
  "baseline_near_or_full_ara_rate": 0.0008583690987124463
}


## 2. Recompute correct daily rank

In [4]:
score_cols = [
    "score_ara",
    "ara_signature_score_v1",
    "final_ara_watch_score_50_50",
    "final_ara_watch_score_60_40",
    "final_ara_watch_score_70_30",
    "score_scalp",
    "score_swing",
    "score_momentum_5d",
    "score_momentum_10d",
    "score_position",
]
score_cols = [c for c in score_cols if c in df.columns]

for c in score_cols:
    df[f"daily_rank_{c}"] = df.groupby("date")[c].rank(method="first", ascending=False)
    df[f"daily_pct_{c}"] = df.groupby("date")[c].rank(method="average", pct=True)

rank_cols = ["date", "ticker", "ara_opening_class", "next_open_ret", "ara_ratio"]
rank_cols += [f"daily_rank_{c}" for c in ["score_ara", "ara_signature_score_v1"] if f"daily_rank_{c}" in df.columns]
rank_cols += ["score_ara", "ara_signature_score_v1"]
display(df.loc[df["is_strong_gap_or_better"].eq(1), [c for c in rank_cols if c in df.columns]].sort_values(["date", "ticker"]))

,date,ticker,ara_opening_class,next_open_ret,ara_ratio,daily_rank_score_ara,daily_rank_ara_signature_score_v1,score_ara,ara_signature_score_v1
414,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,103.0,86.0,0.094538,0.520024
671,2026-05-18,LCKM,strong_gap_up,0.133929,0.382653,104.0,13.0,0.090767,0.755752
559,2026-05-20,INTD,near_ara,0.202797,0.811189,133.0,2.0,0.050528,0.909242
325,2026-05-22,DIVA,strong_gap_up,0.176471,0.504202,1.0,101.0,0.876316,0.458509
1075,2026-05-22,TALF,strong_gap_up,0.102564,0.410256,122.0,2.0,0.051335,0.899410


## 3. Single-score top-k evaluation

In [5]:
def eval_mask(mask, name):
    sub = df.loc[mask].copy()
    baseline_strong = df["is_strong_gap_or_better"].mean()
    baseline_near = df["is_near_or_full_ara"].mean()
    out = {
        "strategy": name,
        "n_candidates": int(len(sub)),
        "n_days": int(sub["date"].nunique()) if len(sub) else 0,
        "avg_next_open_ret": float(sub["next_open_ret"].mean()) if len(sub) else np.nan,
        "median_next_open_ret": float(sub["next_open_ret"].median()) if len(sub) else np.nan,
        "p90_ara_ratio": float(sub["ara_ratio"].quantile(0.9)) if len(sub) else np.nan,
        "strong_gap_rate": float(sub["is_strong_gap_or_better"].mean()) if len(sub) else np.nan,
        "near_or_full_ara_rate": float(sub["is_near_or_full_ara"].mean()) if len(sub) else np.nan,
        "event_capture_strong": int(sub["is_strong_gap_or_better"].sum()) if len(sub) else 0,
        "event_capture_near": int(sub["is_near_or_full_ara"].sum()) if len(sub) else 0,
        "lift_strong_gap_vs_baseline": float(sub["is_strong_gap_or_better"].mean() / baseline_strong) if len(sub) and baseline_strong > 0 else np.nan,
        "lift_near_ara_vs_baseline": float(sub["is_near_or_full_ara"].mean() / baseline_near) if len(sub) and baseline_near > 0 else np.nan,
    }
    return out

rows = []
TOP_KS = [1, 2, 3, 5, 7, 10, 15, 20]
for c in score_cols:
    r = f"daily_rank_{c}"
    if r in df.columns:
        for k in TOP_KS:
            rows.append(eval_mask(df[r] <= k, f"{c}_top{k}") | {"score_col": c, "top_k": k, "method": "single_score_topk"})

single_eval = pd.DataFrame(rows).sort_values(["event_capture_near", "event_capture_strong", "strong_gap_rate", "avg_next_open_ret"], ascending=False)
single_eval.to_csv(OUTPUT_DIR / "single_score_topk_evaluation_v3.csv", index=False)
display(single_eval.head(40))

,strategy,n_candidates,n_days,avg_next_open_ret,median_next_open_ret,p90_ara_ratio,strong_gap_rate,near_or_full_ara_rate,event_capture_strong,event_capture_near,lift_strong_gap_vs_baseline,lift_near_ara_vs_baseline,score_col,top_k,method
14,ara_signature_score_v1_top15,90,6,0.007622,0.000000,0.109789,0.033333,0.011111,3,1,7.766667,12.944444,ara_signature_score_v1,15,single_score_topk
15,ara_signature_score_v1_top20,120,6,0.002540,0.000000,0.109789,0.025000,0.008333,3,1,5.825000,9.708333,ara_signature_score_v1,20,single_score_topk
9,ara_signature_score_v1_top2,12,6,0.030626,0.009799,0.378680,0.166667,0.083333,2,1,38.833333,97.083333,ara_signature_score_v1,2,single_score_topk
10,ara_signature_score_v1_top3,18,6,0.020417,0.000000,0.189219,0.111111,0.055556,2,1,25.888889,64.722222,ara_signature_score_v1,3,single_score_topk
11,ara_signature_score_v1_top5,30,6,0.015598,0.002577,0.116025,0.066667,0.033333,2,1,15.533333,38.833333,ara_signature_score_v1,5,single_score_topk
12,ara_signature_score_v1_top7,42,6,0.017326,0.005702,0.139500,0.047619,0.023810,2,1,11.095238,27.738095,ara_signature_score_v1,7,single_score_topk
13,ara_signature_score_v1_top10,60,6,0.012272,0.004167,0.105666,0.033333,0.016667,2,1,7.766667,19.416667,ara_signature_score_v1,10,single_score_topk
0,score_ara_top1,6,6,0.003783,-0.003546,0.305434,0.166667,0.000000,1,0,38.833333,0.000000,score_ara,1,single_score_topk
1,score_ara_top2,12,6,-0.005778,0.000000,0.121239,0.083333,0.000000,1,0,19.416667,0.000000,score_ara,2,single_score_topk
2,score_ara_top3,18,6,-0.014094,0.000000,0.130479,0.055556,0.000000,1,0,12.944444,0.000000,score_ara,3,single_score_topk


## 4. Two-lane OR strategy evaluation

In [6]:
# Two-lane logic:
# lane A: score_ara top-k
# lane B: ara_signature_score_v1 top-k
# final candidate = lane A OR lane B

rows = []
for score_k in [1, 2, 3, 5]:
    for sig_k in [1, 2, 3, 5, 7, 10, 15, 20]:
        mask_model = df["daily_rank_score_ara"] <= score_k
        mask_sig = df["daily_rank_ara_signature_score_v1"] <= sig_k
        mask = mask_model | mask_sig
        rows.append(
            eval_mask(mask, f"score_ara_top{score_k}_OR_signature_top{sig_k}")
            | {
                "model_top_k": score_k,
                "signature_top_k": sig_k,
                "method": "two_lane_or",
            }
        )

two_lane_eval = pd.DataFrame(rows).sort_values(
    ["event_capture_near", "event_capture_strong", "strong_gap_rate", "avg_next_open_ret"],
    ascending=False,
)
two_lane_eval.to_csv(OUTPUT_DIR / "two_lane_or_strategy_evaluation_v3.csv", index=False)
display(two_lane_eval.head(40))

,strategy,n_candidates,n_days,avg_next_open_ret,median_next_open_ret,p90_ara_ratio,strong_gap_rate,near_or_full_ara_rate,event_capture_strong,event_capture_near,lift_strong_gap_vs_baseline,lift_near_ara_vs_baseline,model_top_k,signature_top_k,method
6,score_ara_top1_OR_signature_top15,96,6,0.007382,0.000000,0.111788,0.041667,0.010417,4,1,9.708333,12.135417,1,15,two_lane_or
14,score_ara_top2_OR_signature_top15,102,6,0.006046,0.000000,0.114778,0.039216,0.009804,4,1,9.137255,11.421569,2,15,two_lane_or
22,score_ara_top3_OR_signature_top15,108,6,0.004003,0.000000,0.119230,0.037037,0.009259,4,1,8.629630,10.787037,3,15,two_lane_or
30,score_ara_top5_OR_signature_top15,120,6,0.002048,0.000000,0.115156,0.033333,0.008333,4,1,7.766667,9.708333,5,15,two_lane_or
7,score_ara_top1_OR_signature_top20,125,6,0.003792,0.000000,0.112287,0.032000,0.008000,4,1,7.456000,9.320000,1,20,two_lane_or
15,score_ara_top2_OR_signature_top20,131,6,0.002916,0.000000,0.114833,0.030534,0.007634,4,1,7.114504,8.893130,2,20,two_lane_or
23,score_ara_top3_OR_signature_top20,136,6,0.002400,0.000000,0.120007,0.029412,0.007353,4,1,6.852941,8.566176,3,20,two_lane_or
31,score_ara_top5_OR_signature_top20,148,6,0.000945,0.000000,0.115802,0.027027,0.006757,4,1,6.297297,7.871622,5,20,two_lane_or
1,score_ara_top1_OR_signature_top2,18,6,0.021678,0.005911,0.438440,0.166667,0.055556,3,1,38.833333,64.722222,1,2,two_lane_or
2,score_ara_top1_OR_signature_top3,24,6,0.016259,0.000000,0.319179,0.125000,0.041667,3,1,29.125000,48.541667,1,3,two_lane_or


## 5. Event capture table by strategy

In [7]:
# Pilih beberapa policy kandidat untuk dilihat per event.
POLICIES = {
    "score_ara_top1": df["daily_rank_score_ara"] <= 1,
    "signature_top3": df["daily_rank_ara_signature_score_v1"] <= 3,
    "signature_top15": df["daily_rank_ara_signature_score_v1"] <= 15,
    "two_lane_score1_sig3": (df["daily_rank_score_ara"] <= 1) | (df["daily_rank_ara_signature_score_v1"] <= 3),
    "two_lane_score1_sig15": (df["daily_rank_score_ara"] <= 1) | (df["daily_rank_ara_signature_score_v1"] <= 15),
}

events = df.loc[df["is_strong_gap_or_better"].eq(1)].copy()
for name, mask in POLICIES.items():
    events[f"captured_by_{name}"] = mask.loc[events.index].values

event_cols = [
    "date", "ticker", "ara_opening_class", "next_open_ret", "ara_ratio",
    "score_ara", "ara_signature_score_v1",
    "daily_rank_score_ara", "daily_rank_ara_signature_score_v1",
]
event_cols += [c for c in events.columns if c.startswith("captured_by_")]

events[event_cols].sort_values(["date", "ticker"]).to_csv(OUTPUT_DIR / "event_capture_by_two_lane_policy_v3.csv", index=False)
display(events[event_cols].sort_values(["date", "ticker"]))

,date,ticker,ara_opening_class,next_open_ret,ara_ratio,score_ara,ara_signature_score_v1,daily_rank_score_ara,daily_rank_ara_signature_score_v1,captured_by_score_ara_top1,captured_by_signature_top3,captured_by_signature_top15,captured_by_two_lane_score1_sig3,captured_by_two_lane_score1_sig15
414,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,0.094538,0.520024,103.0,86.0,False,False,False,False,False
671,2026-05-18,LCKM,strong_gap_up,0.133929,0.382653,0.090767,0.755752,104.0,13.0,False,False,True,False,True
559,2026-05-20,INTD,near_ara,0.202797,0.811189,0.050528,0.909242,133.0,2.0,False,True,True,True,True
325,2026-05-22,DIVA,strong_gap_up,0.176471,0.504202,0.876316,0.458509,1.0,101.0,True,False,False,True,True
1075,2026-05-22,TALF,strong_gap_up,0.102564,0.410256,0.051335,0.899410,122.0,2.0,False,True,True,True,True


## 6. Build daily two-lane watchlist

In [8]:
# Default research policy from v3:
# - ARA model lane: score_ara top 1
# - Behavioral lane: signature top 3
# This intentionally keeps list small and captures DIVA-type + INTD/TALF-type events in the short sample.

MODEL_TOP_K = 1
SIGNATURE_TOP_K = 3

watch = df[
    (df["daily_rank_score_ara"] <= MODEL_TOP_K)
    | (df["daily_rank_ara_signature_score_v1"] <= SIGNATURE_TOP_K)
].copy()

def assign_lane(row):
    m = row["daily_rank_score_ara"] <= MODEL_TOP_K
    s = row["daily_rank_ara_signature_score_v1"] <= SIGNATURE_TOP_K
    if m and s:
        return "dual_confirm"
    if m:
        return "ara_model_lane"
    if s:
        return "behavioral_ignition_lane"
    return "none"

watch["ara_watch_lane"] = watch.apply(assign_lane, axis=1)

watch_cols = [
    "date", "ticker", "ara_watch_lane",
    "daily_rank_score_ara", "score_ara",
    "daily_rank_ara_signature_score_v1", "ara_signature_score_v1",
    "next_open_ret", "ara_ratio", "ara_opening_class",
]
extra_cols = [
    "volume_ratio_20d", "ret_1d", "ret_5d", "close_vs_ma20",
    "volatility_20d", "buyer_dominance_ratio", "net_flow_ratio",
    "rank1_same_buyer_streak", "rank1_buyer"
]
watch_cols += [c for c in extra_cols if c in watch.columns]

watch[watch_cols].sort_values(["date", "ara_watch_lane", "daily_rank_score_ara", "daily_rank_ara_signature_score_v1"]).to_csv(
    OUTPUT_DIR / "two_lane_daily_ara_watchlist_v3.csv", index=False
)
display(watch[watch_cols].sort_values(["date", "ara_watch_lane", "daily_rank_score_ara", "daily_rank_ara_signature_score_v1"]))

,date,ticker,ara_watch_lane,daily_rank_score_ara,score_ara,daily_rank_ara_signature_score_v1,ara_signature_score_v1,next_open_ret,ara_ratio,ara_opening_class,volume_ratio_20d,ret_1d,ret_5d,close_vs_ma20,volatility_20d,buyer_dominance_ratio,net_flow_ratio,rank1_same_buyer_streak,rank1_buyer
787,2026-05-13,MSJA,ara_model_lane,1.0,0.922260,137.0,0.410024,0.000000,0.000000,normal_movement,1.117743,-0.203810,-0.203810,-0.192894,0.051810,0.504710,0.000000,1,AZ
620,2026-05-13,KJEN,behavioral_ignition_lane,83.0,0.142883,2.0,0.880024,0.000000,0.000000,normal_movement,6.211097,0.244444,0.200000,0.267446,0.104825,0.387485,-0.000272,3,XL
348,2026-05-13,DPUM,behavioral_ignition_lane,87.0,0.136516,1.0,0.897188,0.026042,0.074405,normal_movement,1.810816,0.600000,0.573770,0.484345,0.140781,0.334963,0.003846,3,XL
727,2026-05-13,MEDS,behavioral_ignition_lane,98.0,0.105766,3.0,0.864207,0.000000,0.000000,normal_movement,3.007394,0.204545,0.358974,0.407703,0.068983,0.265050,0.001891,8,XL
271,2026-05-18,CUAN,ara_model_lane,1.0,0.856928,166.0,0.353107,0.026667,0.106667,normal_movement,1.649178,-0.117647,-0.375000,-0.436302,0.089242,0.229790,-0.006577,2,AK
589,2026-05-18,KAEF,behavioral_ignition_lane,61.0,0.145616,2.0,0.894879,0.023622,0.094488,normal_movement,8.550050,0.058333,0.328452,0.255934,0.051902,0.423289,0.002326,1,XL
728,2026-05-18,MEDS,behavioral_ignition_lane,77.0,0.123825,1.0,0.937718,0.025641,0.073260,normal_movement,7.017251,0.103774,0.519481,0.502890,0.070431,0.412437,0.004855,9,XL
665,2026-05-18,LABS,behavioral_ignition_lane,106.0,0.090631,3.0,0.887500,0.000000,0.000000,normal_movement,15.562257,0.090909,0.306122,0.222151,0.047081,0.486680,0.000041,1,XL
203,2026-05-19,BUMI,ara_model_lane,1.0,0.875657,139.0,0.415217,-0.026882,-0.076805,normal_movement,2.847931,-0.097087,-0.191304,-0.198621,0.040269,0.103246,0.009362,1,XL
52,2026-05-19,ASPR,behavioral_ignition_lane,72.0,0.171300,2.0,0.890580,0.012931,0.051724,normal_movement,3.119724,0.154229,0.397590,0.630358,0.102772,0.403534,-0.001832,3,XL


## 7. Summary

In [9]:
recommended_policy = (df["daily_rank_score_ara"] <= 1) | (df["daily_rank_ara_signature_score_v1"] <= 3)
recommended_metrics = eval_mask(recommended_policy, "recommended_two_lane_score1_sig3")

summary_v3 = {
    "input_dataset": str(DATASET_PATH),
    "rows": int(len(df)),
    "unique_dates": int(df["date"].nunique()),
    "unique_tickers": int(df["ticker"].nunique()),
    "class_distribution": df["ara_opening_class"].value_counts(dropna=False).to_dict(),
    "baseline": {
        "strong_gap_rate": float(df["is_strong_gap_or_better"].mean()),
        "near_or_full_ara_rate": float(df["is_near_or_full_ara"].mean()),
    },
    "recommended_policy": {
        "description": "Two-lane OR: score_ara top 1 OR ara_signature_score_v1 top 3 per day",
        "model_top_k": 1,
        "signature_top_k": 3,
        "metrics": recommended_metrics,
    },
    "key_warning": "Sample is still very small. Treat this as research note / hypothesis, not production proof.",
}
with open(OUTPUT_DIR / "research_summary_v3.json", "w", encoding="utf-8") as f:
    json.dump(summary_v3, f, indent=2, ensure_ascii=False, default=str)

print(json.dumps(summary_v3, indent=2, ensure_ascii=False, default=str))

{
  "input_dataset": "ara_opening_signature_research_v2_outputs/ara_opening_signature_dataset_v2.csv",
  "rows": 1165,
  "unique_dates": 6,
  "unique_tickers": 259,
  "class_distribution": {
    "normal_movement": 1160,
    "strong_gap_up": 4,
    "near_ara": 1
  },
  "baseline": {
    "strong_gap_rate": 0.004291845493562232,
    "near_or_full_ara_rate": 0.0008583690987124463
  },
  "recommended_policy": {
    "description": "Two-lane OR: score_ara top 1 OR ara_signature_score_v1 top 3 per day",
    "model_top_k": 1,
    "signature_top_k": 3,
    "metrics": {
      "strategy": "recommended_two_lane_score1_sig3",
      "n_candidates": 24,
      "n_days": 6,
      "avg_next_open_ret": 0.01625863043970449,
      "median_next_open_ret": 0.0,
      "p90_ara_ratio": 0.3191794871794871,
      "strong_gap_rate": 0.125,
      "near_or_full_ara_rate": 0.041666666666666664,
      "event_capture_strong": 3,
      "event_capture_near": 1,
      "lift_strong_gap_vs_baseline": 29.125,
      "lift_nea